In [1]:
import tellurium as te
import numpy as np
import pandas as pd
import os
import json
import logging
from datetime import datetime

In [2]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
ANTIMONY_MODEL = r'iMC057.txt'

OUTDIR = "perfect_datasets"

# Metabolites to vary (discrete grids)
VARY_METABOLITES = {
    "ATP": (0, 2, 0.1),
    "AcetylCoA": (0, 2, 0.1),
    "L_Serine": (0, 2, 0.1),
    "NADH": (0, 2, 0.1),
    "AMP": (0, 2, 0.1),
    "HCO3": (0, 10, 0.5),
    "Hydroxycitrate": (0, 10, 0.5),
    "NAD": (0, 2, 0.1),
    "NADPH": (0, 2, 0.1),
    "NADP": (0, 2, 0.1),
    "Pyruvate": (0, 2, 0.1),
    "SMalate": (0, 2, 0.1),
    "ADP": (0, 2, 0.1),
    "CoA": (0, 2, 0.1),
    "Oxaloacetate": (0, 2, 0.1),
    "Formate": (0, 2, 0.1),
    "hEC6411": (0, 0.05, 0.005),
    "hEC43117": (0, 0.05, 0.005),
    "eEC11137": (0.01, 0.05, 0.001)
}

# Quantities
N_HIGH = 100
N_MED = 50
N_LOW = 25
N_ONE = 1

# Simulate for 4 hours, record every minute
SIM_TIME_HOURS = 4
TIME_UNITS_PER_HOUR = 60.0  # assume model uses minutes
SAMPLE_INTERVAL_MINUTES = 1

# Random seed
SEED = 12345
np.random.seed(SEED)

In [4]:
def load_model():
    """Load the model and return RoadRunner object + initial concentrations."""
    if not ANTIMONY_MODEL:
        raise ValueError("Please set ANTIMONY_MODEL to your Antimony file path.")
    rr = te.loada(ANTIMONY_MODEL)
    species_ids = list(rr.getFloatingSpeciesIds())
    initial_vals = {sid: float(rr[sid]) for sid in species_ids}
    return rr, initial_vals

def generate_high_starting_conditions(initial_vals, vary_mets, n_high):
    """Generate n_high random starting conditions on a discrete grid."""
    starts = []
    grid_choices = {met: np.arange(mn, mx + 1e-9, step) for met, (mn, mx, step) in vary_mets.items()}

    for i in range(n_high):
        sc = dict(initial_vals)
        for met, grid in grid_choices.items():
            if met not in sc:
                raise KeyError(f"Metabolite '{met}' not found in model species.")
            sc[met] = float(np.random.choice(grid))
        sc['_exp_id'] = f"exp_{i+1:03d}"
        starts.append(sc)
    return starts

def simulate_start(rr, initial_cond, sim_time_hr, dt_min):
    """Simulate model from given initial condition, record each minute."""
    for sid, val in initial_cond.items():
        if sid == '_exp_id':
            continue
        rr[sid] = float(val)

    start_t = 0
    end_t = sim_time_hr * TIME_UNITS_PER_HOUR  # convert to minutes
    n_points = int(end_t / dt_min) + 1
    sim = rr.simulate(start_t, end_t, n_points)

    # Convert to DataFrame
    colnames = ['time'] + list(rr.getFloatingSpeciesIds())
    df = pd.DataFrame(sim, columns=colnames)
    return df


def main():
    os.makedirs(OUTDIR, exist_ok=True)
    rr, default_initials = load_model()
    model_species = set(default_initials.keys())

    # check vary metabolites
    for met in VARY_METABOLITES.keys():
        if met not in model_species:
            raise KeyError(f"Vary metabolite '{met}' not found in model species. "
                           f"Available: {sorted(model_species)[:20]}")

    measured_species = list(model_species)  # record all

    # Generate starting conditions
    high_starts = generate_high_starting_conditions(default_initials, VARY_METABOLITES, N_HIGH)
    med_indices = np.random.choice(range(N_HIGH), size=N_MED, replace=False)
    low_indices = np.random.choice(range(N_MED), size=N_LOW, replace=False)
    medium_starts = [high_starts[i] for i in med_indices]
    low_starts = [medium_starts[i] for i in low_indices]
    one_starts = [dict(default_initials, _exp_id="exp_001")]

    meta = {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "seed": int(SEED),
        "vary_metabolites": VARY_METABOLITES,
        "n_high": N_HIGH,
        "n_med": N_MED,
        "n_low": N_LOW,
        "n_one": N_ONE,
        "measured_species": measured_species,
        "time_resolution_min": SAMPLE_INTERVAL_MINUTES,
        "sim_time_hr": SIM_TIME_HOURS,
        "no_noise": True
    }

    with open(os.path.join(OUTDIR, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    def run_and_save(starting_conditions, label_prefix):
        dfs = []
        for sc in starting_conditions:
            exp_id = sc['_exp_id']
            logging.info(f"Simulating {exp_id} ({label_prefix})")
            df = simulate_start(rr, sc, SIM_TIME_HOURS, SAMPLE_INTERVAL_MINUTES)
            df.insert(0, "exp_id", exp_id)
            dfs.append(df)

        all_df = pd.concat(dfs, ignore_index=True)
        all_df.to_csv(os.path.join(OUTDIR, f"{label_prefix}_timeseries.csv"), index=False)

        # also save starting conditions
        starts_flat = []
        for sc in starting_conditions:
            sc_flat = {k: v for k, v in sc.items() if k != '_exp_id'}
            sc_flat['exp_id'] = sc['_exp_id']
            starts_flat.append(sc_flat)
        pd.DataFrame(starts_flat).to_csv(os.path.join(OUTDIR, f"{label_prefix}_starting_conditions.csv"), index=False)
        return all_df

    # Run all quantity levels
    run_and_save(one_starts, "oneQ")
    run_and_save(low_starts, "lowQ")
    run_and_save(medium_starts, "medQ")
    run_and_save(high_starts, "highQ")

    logging.info(f"All datasets generated in {OUTDIR}")

In [5]:
if __name__ == "__main__":
    main()

C:\Users\mkcoo\AppData\Local\Temp\ipykernel_12688\3875288930.py:65: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).

2025-10-07 13:52:31,809 INFO: Simulating exp_001 (oneQ)
2025-10-07 13:52:31,918 INFO: Simulating exp_074 (lowQ)
2025-10-07 13:52:31,965 INFO: Simulating exp_061 (lowQ)
2025-10-07 13:52:32,016 INFO: Simulating exp_034 (lowQ)
2025-10-07 13:52:32,055 INFO: Simulating exp_023 (lowQ)
2025-10-07 13:52:32,093 INFO: Simulating exp_012 (lowQ)
2025-10-07 13:52:32,135 INFO: Simulating exp_031 (lowQ)
2025-10-07 13:52:32,171 INFO: Simulating exp_042 (lowQ)
2025-10-07 13:52:32,211 INFO: Simulating exp_073 (lowQ)
2025-10-07 13:52:32,254 INFO: Simulating exp_010 (lowQ)
2025-10-07 13:52:32,368 INFO: Simulating exp_047 (lowQ)
2025-10-07 13:52:32,420 INFO: Simulating exp_059 (lowQ)
2025-10-07 13:52:32,461 INFO: Simulating exp_025 (lowQ)

In [ ]:
samples = [
    "200X_mdh_neg",
    "200X_pyc_neg",
    "200X_fdh_neg",
    "200X_mdh",
    "200X_fdh",
    "200X_pyc",
    "200X_sds",
    "Std_0",
    "Std_1",
    "Std_2",
    "Std_3",
    "Std_4"
]

std_indices = [0, 1, 2, 4]   # your Std_0–Std_4 groups
rep_indices = [1, 2, 3]         # triplicates

output = []

for sample in samples:
    for std in std_indices:
        for rep in rep_indices:
            output.append(f"{sample}_{std}_{rep}")

# print as one Excel-ready column
("\n".join(output))


200X_mdh_neg_0_1
200X_mdh_neg_0_2
200X_mdh_neg_0_3
200X_mdh_neg_1_1
200X_mdh_neg_1_2
200X_mdh_neg_1_3
200X_mdh_neg_2_1
200X_mdh_neg_2_2
200X_mdh_neg_2_3
200X_mdh_neg_4_1
200X_mdh_neg_4_2
200X_mdh_neg_4_3
200X_pyc_neg_0_1
200X_pyc_neg_0_2
200X_pyc_neg_0_3
200X_pyc_neg_1_1
200X_pyc_neg_1_2
200X_pyc_neg_1_3
200X_pyc_neg_2_1
200X_pyc_neg_2_2
200X_pyc_neg_2_3
200X_pyc_neg_4_1
200X_pyc_neg_4_2
200X_pyc_neg_4_3
200X_fdh_neg_0_1
200X_fdh_neg_0_2
200X_fdh_neg_0_3
200X_fdh_neg_1_1
200X_fdh_neg_1_2
200X_fdh_neg_1_3
200X_fdh_neg_2_1
200X_fdh_neg_2_2
200X_fdh_neg_2_3
200X_fdh_neg_4_1
200X_fdh_neg_4_2
200X_fdh_neg_4_3
200X_mdh_0_1
200X_mdh_0_2
200X_mdh_0_3
200X_mdh_1_1
200X_mdh_1_2
200X_mdh_1_3
200X_mdh_2_1
200X_mdh_2_2
200X_mdh_2_3
200X_mdh_4_1
200X_mdh_4_2
200X_mdh_4_3
200X_fdh_0_1
200X_fdh_0_2
200X_fdh_0_3
200X_fdh_1_1
200X_fdh_1_2
200X_fdh_1_3
200X_fdh_2_1
200X_fdh_2_2
200X_fdh_2_3
200X_fdh_4_1
200X_fdh_4_2
200X_fdh_4_3
200X_pyc_0_1
200X_pyc_0_2
200X_pyc_0_3
200X_pyc_1_1
200X_pyc_1_2
200X_pyc_1_